In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
fig_small = (7, 4)
fig_medium = (10, 6)
fig_big = (15, 10)

In [ ]:
plt.style.use("default")  # reset to Matplotlib default (light)
mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.transparent": False,
})

In [ ]:
# dataset_name = "dataset_cifar_large_hyp_rand"
# dataset_name = "dataset_cifar_large_hyp_fix"
# dataset_name = "dataset_cifar_large_seed"

# dataset_name = "dataset_mnist_hyp_rand"
dataset_name = "tune_zoo_mnist_hyperparameter_10_fixed_seeds"
# dataset_name = "dataset_mnist_seed"

out_dir = Path(f"plots/{dataset_name}")
out_dir.mkdir(parents=True, exist_ok=True)

## Creat complete Data Frame
Create a dataframe based on all result.json files inside the raw model zoo.

In [ ]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Read a JSONL file (one JSON object per line)."""
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on {path} line {line_no}: {e}") from e
    return rows


def build_long_dataframe(root_dir: str | Path) -> pd.DataFrame:
    """
    Builds a dataframe with one row per JSON line, plus:
      - dataset_name, model_name inferred from path parts
      - file_path for traceability
      - config flattened into columns (e.g., 'config.model::type')
    """
    root = Path(root_dir)
    records = []

    for result_path in root.rglob("result.json"):
        # /dataset_name/model_name/result.json
        model_name_raw = result_path.parent.name
        dataset_name = result_path.parent.parent.name

        # keep only prefix before the first '::'
        model_name = model_name_raw.split("::", 1)[0]

        for row in read_jsonl(result_path):
            row = dict(row)
            row["dataset_name"] = dataset_name
            row["model_name"] = model_name
            row["model_name_full"] = model_name_raw  # optional but useful
            row["file_path"] = str(result_path)
            records.append(row)

    return pd.json_normalize(records, sep=".") if records else pd.DataFrame()

In [ ]:
df_long = build_long_dataframe(f"../../data/{dataset_name}")

Define metrices and column names for all further cells

In [ ]:
epoch_col = "training_iteration"
model_col = "model_name"

metrics = [
    "train_loss", "validation_loss", "test_loss",
    "train_acc", "validation_acc", "test_acc",
]

EPOCH_RANGE = (1, 50)  # set to None if you want "all epochs that exist globally"

Clean the dataframe
- remove models with nan values
- remove models with missing epochs

In [ ]:
df_long = df_long.copy()
df_long[epoch_col] = pd.to_numeric(df_long[epoch_col], errors="coerce")

for c in metrics:
    if c in df_long.columns:
        df_long[c] = pd.to_numeric(df_long[c], errors="coerce").replace([-999, -999.0], np.nan)

# ---- 2) drop rows with any NaN in epoch/model/metrics ----
keep_cols = [model_col, epoch_col] + [c for c in metrics if c in df_long.columns]
df_long = df_long.dropna(subset=keep_cols)

# ---- 3) drop models with missing epochs (choose your epoch range) ----
if EPOCH_RANGE is not None:
    e_min, e_max = EPOCH_RANGE
    expected_epochs = set(range(e_min, e_max + 1))

    # keep only epochs in range
    df_long = df_long[(df_long[epoch_col] >= e_min) & (df_long[epoch_col] <= e_max)]

    # models that have all expected epochs
    epochs_per_model = df_long.groupby(model_col)[epoch_col].apply(lambda s: set(s.astype(int)))
    good_models = epochs_per_model.index[epochs_per_model.apply(lambda s: expected_epochs.issubset(s))]

    df_long = df_long[df_long[model_col].isin(good_models)].copy()

# optional: enforce one row per (model, epoch) if duplicates exist
df_long = df_long.sort_values([model_col, epoch_col]).drop_duplicates([model_col, epoch_col], keep="last")
n_models = df_long[model_col].nunique()
print("Cleaned df_long:", df_long.shape)
print("n_models:", n_models)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
display(df_long.head(5))

## Build basic statistics for model zoo

In [ ]:
# ensure numeric
dfm = df_long.copy()
dfm[metrics] = dfm[metrics].apply(pd.to_numeric, errors="coerce")
dfm[epoch_col] = pd.to_numeric(dfm[epoch_col], errors="coerce")

summary = (
    dfm.groupby(epoch_col)[metrics]
    .agg(["mean", "std", "min", "max"])
    .sort_index()
)

display(summary)

## Analysis

Std. of Accuracy over epochs

In [ ]:
x = summary.index.to_numpy()
y = summary[("test_acc", "std")].to_numpy()

plt.figure(figsize=(10, 5), dpi=180)
plt.plot(x, y, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Std (test_acc)")
plt.title(f"Std of test_acc over epochs (N={n_models})")
plt.grid(True)
plt.tight_layout()
plt.show()

Display top 500 Models

In [ ]:
K = 500
metric_key = "test_acc"

# --- ensure numeric ---
dfm = df_long.copy()
dfm[epoch_col] = pd.to_numeric(dfm[epoch_col], errors="coerce")
dfm[metric_key] = pd.to_numeric(dfm[metric_key], errors="coerce")

dfm[metric_key] = dfm[metric_key].replace([-999, -999.0], np.nan)

# ----- final epoch -----
final_epoch = int(dfm[epoch_col].max())

# ----- get final-epoch score per model (one row per model) -----
final_rows = dfm[dfm[epoch_col] == final_epoch].copy()

# if there can be duplicates per model at the final epoch, take the max (or mean)
final_scores = (
    final_rows.groupby("model_name")[metric_key]
    .max()
    .sort_values(ascending=False)
)

top_models = final_scores.head(K).index.tolist()

# ----- build a model x epoch matrix (like your acc_mat) -----
acc_mat = (
    dfm[dfm["model_name"].isin(top_models)]
    .pivot_table(index="model_name", columns=epoch_col, values=metric_key, aggfunc="max")
    .sort_index(axis=1)
)

# ----- plot curves for top-K -----
short_labels = {m: f"m{i + 1:03d}" for i, m in enumerate(top_models)}

plt.figure(figsize=(12, 6), dpi=180)
for m in top_models:
    plt.plot(acc_mat.columns.to_numpy(), acc_mat.loc[m].to_numpy(), label=short_labels[m])

plt.xlabel("Epoch")
plt.ylabel(metric_key)
plt.title(f"{metric_key} over epochs for top-{K} models (by final epoch {final_epoch}, N={n_models})")
plt.grid(True)
plt.tight_layout()
plt.show()

# ----- display top-K table -----
top_table = pd.DataFrame({
    "short": [short_labels[m] for m in top_models],
    "model": top_models,
    "final_acc": final_scores.loc[top_models].to_numpy(),
}).reset_index(drop=True)

display(top_table)

# ----- epoch stats across the top-K models -----
std_per_epoch = acc_mat.std(axis=0, ddof=1)
mean_per_epoch = acc_mat.mean(axis=0)
min_per_epoch = acc_mat.min(axis=0)
max_per_epoch = acc_mat.max(axis=0)

epoch_stats = pd.DataFrame({
    "epoch": acc_mat.columns.astype(int),
    "mean": mean_per_epoch.to_numpy(),
    "std": std_per_epoch.to_numpy(),
    "min": min_per_epoch.to_numpy(),
    "max": max_per_epoch.to_numpy(),
}).set_index("epoch")

display(epoch_stats.round(4))

In [ ]:
# std over epochs across the top-K models (acc_mat is models x epochs)
std_per_epoch = acc_mat.std(axis=0, ddof=1).sort_index()

plt.figure(figsize=fig_big, dpi=180)
plt.plot(std_per_epoch.index.to_numpy(), std_per_epoch.to_numpy(), linewidth=2)
plt.xlabel("Epoch")
plt.ylabel(f"{metric_key} std")
plt.title(f"Std of {metric_key} over epochs (top-{K} models)")
plt.grid(True)
plt.tight_layout()

# optional save
plt.savefig(out_dir / f"2_topK_std_{metric_key}_epoch{final_epoch}_K{K}.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ---------- settings ----------
K_worst = 200
metric_key = "test_acc"  # change if needed

# assumes dfm already exists from before; if not:
dfm = df_long.copy()
dfm[epoch_col] = pd.to_numeric(dfm[epoch_col], errors="coerce")
dfm[metric_key] = pd.to_numeric(dfm[metric_key], errors="coerce")
dfm[metric_key] = dfm[metric_key].replace([-999, -999.0], np.nan)  # optional sentinel

# ---------- final epoch + worst-K selection ----------
final_epoch = int(dfm[epoch_col].max())

final_rows = dfm[dfm[epoch_col] == final_epoch].copy()
final_scores = (
    final_rows.groupby("model_name")[metric_key]
    .max()  # or mean(), depending on duplicates
    .dropna()
    .sort_values(ascending=True)  # ascending => worst first
)

worst_models = final_scores.head(K_worst).index.tolist()

# models x epochs matrix for worst-K
acc_worst = (
    dfm[dfm["model_name"].isin(worst_models)]
    .pivot_table(index="model_name", columns=epoch_col, values=metric_key, aggfunc="max")
    .sort_index(axis=1)
)

# ---------- plot accuracy curves for worst-K ----------
short_labels_w = {m: f"w{i + 1:03d}" for i, m in enumerate(worst_models)}

plt.figure(figsize=fig_big, dpi=180)
for m in worst_models:
    plt.plot(acc_worst.columns.to_numpy(), acc_worst.loc[m].to_numpy(), label=short_labels_w[m])

plt.xlabel("Epoch")
plt.ylabel(metric_key)
plt.title(f"{metric_key} over epochs for worst-{K_worst} models (by final epoch {final_epoch})")
plt.grid(True)
plt.tight_layout()

# optional save
plt.savefig(out_dir / f"worstK_{metric_key}_epoch{final_epoch}_K{K_worst}.pdf", bbox_inches="tight")
plt.show()

# ---------- table of worst models ----------
display(pd.DataFrame({
    "short": [short_labels_w[m] for m in worst_models],
    "model": worst_models,
    "final_acc": final_scores.loc[worst_models].to_numpy(),
}).reset_index(drop=True).round(4))

# ---------- std over epochs for worst-K ----------
std_per_epoch_worst = acc_worst.std(axis=0, ddof=1).sort_index()

In [ ]:
plt.figure(figsize=fig_big, dpi=180)
plt.plot(std_per_epoch_worst.index.to_numpy(), std_per_epoch_worst.to_numpy(), linewidth=2)
plt.xlabel("Epoch")
plt.ylabel(f"{metric_key} std")
plt.title(f"Std of {metric_key} over epochs (worst-{K_worst} models)")
plt.grid(True)
plt.tight_layout()

# optional save
plt.savefig(out_dir / f"worstK_std_{metric_key}_epoch{final_epoch}_K{K_worst}.pdf", bbox_inches="tight")
plt.show()

Final Accuracy above a certain treshold

In [ ]:
only_models_above = 0.2  # keep models with final epoch metric > this
metric_key = "test_acc"  # or "validation_acc", etc.

# --- ensure numeric ---
dfm = df_long.copy()
dfm[epoch_col] = pd.to_numeric(dfm[epoch_col], errors="coerce")
dfm[metric_key] = pd.to_numeric(dfm[metric_key], errors="coerce")
dfm[metric_key] = dfm[metric_key].replace([-999, -999.0], np.nan)  # optional sentinel

n_models = dfm["model_name"].nunique()

# ----- final epoch -----
final_epoch = int(dfm[epoch_col].max())

# ----- final-epoch score per model -----
final_rows = dfm[dfm[epoch_col] == final_epoch].copy()

# if duplicates at final epoch exist, pick max (or mean)
final_scores = (
    final_rows.groupby("model_name")[metric_key]
    .max()
    .dropna()
)

# ----- select models above threshold (sorted desc) -----
selected_models = (
    final_scores[final_scores > only_models_above]
    .sort_values(ascending=False)
    .index
    .tolist()
)

# ----- build models x epochs matrix for selected models -----
acc_sel = (
    dfm[dfm["model_name"].isin(selected_models)]
    .pivot_table(index="model_name", columns=epoch_col, values=metric_key, aggfunc="max")
    .sort_index(axis=1)
)

# ----- plot curves -----
short_labels = {m: f"m{i + 1:03d}" for i, m in enumerate(selected_models)}

plt.figure(figsize=fig_big, dpi=240)
for m in selected_models:
    plt.plot(acc_sel.columns.to_numpy(), acc_sel.loc[m].to_numpy(), label=short_labels[m])

plt.xlabel("Epoch")
plt.ylabel(metric_key)
plt.title(
    f"{metric_key} over epochs for models with final {metric_key} > {only_models_above} "
    f"(final epoch {final_epoch}, n={len(selected_models)} of {n_models})"
)
plt.grid(True)
plt.tight_layout()

plt.savefig(
    out_dir / f"threshold_{metric_key}_epoch{final_epoch}_thr{only_models_above}.pdf",
    bbox_inches="tight"
)
plt.show()

# ----- table of selected models -----
display(pd.DataFrame({
    "short": [short_labels[m] for m in selected_models],
    "model": selected_models,
    f"final_{metric_key}": final_scores.loc[selected_models].to_numpy(),
}).round(4).reset_index(drop=True))

# ----- epoch stats over the selected models -----
epoch_stats = pd.DataFrame({
    "epoch": acc_sel.columns.astype(int),
    "mean": acc_sel.mean(axis=0).to_numpy(),
    "std": acc_sel.std(axis=0, ddof=1).to_numpy(),
    "min": acc_sel.min(axis=0).to_numpy(),
    "max": acc_sel.max(axis=0).to_numpy(),
}).set_index("epoch")

display(epoch_stats.round(4))

In [ ]:
# ----- optional: plot std over epochs for selected models -----
std_per_epoch = acc_sel.std(axis=0, ddof=1).sort_index()

plt.figure(figsize=fig_big, dpi=180)
plt.plot(std_per_epoch.index.to_numpy(), std_per_epoch.to_numpy(), linewidth=2)
plt.xlabel("Epoch")
plt.ylabel(f"{metric_key} std")
plt.title(f"Std of {metric_key} over epochs (final {metric_key} > {only_models_above})")
plt.grid(True)
plt.tight_layout()

plt.savefig(out_dir / f"threshold_std_{metric_key}_epoch{final_epoch}_thr{only_models_above}.pdf",
            bbox_inches="tight")
plt.show()

In [ ]:
only_models_below = 0.2  # keep models with final epoch metric > this
metric_key = "test_acc"  # or "validation_acc", etc.

# --- ensure numeric ---
dfm = df_long.copy()
dfm[epoch_col] = pd.to_numeric(dfm[epoch_col], errors="coerce")
dfm[metric_key] = pd.to_numeric(dfm[metric_key], errors="coerce")
dfm[metric_key] = dfm[metric_key].replace([-999, -999.0], np.nan)  # optional sentinel

n_models = dfm["model_name"].nunique()

# ----- final epoch -----
final_epoch = int(dfm[epoch_col].max())

# ----- final-epoch score per model -----
final_rows = dfm[dfm[epoch_col] == final_epoch].copy()

# if duplicates at final epoch exist, pick max (or mean)
final_scores = (
    final_rows.groupby("model_name")[metric_key]
    .max()
    .dropna()
)

# ----- select models above threshold (sorted desc) -----
selected_models = (
    final_scores[final_scores < only_models_below]
    .sort_values(ascending=True)
    .index
    .tolist()
)

# ----- build models x epochs matrix for selected models -----
acc_sel = (
    dfm[dfm["model_name"].isin(selected_models)]
    .pivot_table(index="model_name", columns=epoch_col, values=metric_key, aggfunc="max")
    .sort_index(axis=1)
)

# ----- plot curves -----
short_labels = {m: f"m{i + 1:03d}" for i, m in enumerate(selected_models)}

plt.figure(figsize=fig_big, dpi=240)
for m in selected_models:
    plt.plot(acc_sel.columns.to_numpy(), acc_sel.loc[m].to_numpy(), label=short_labels[m])

plt.xlabel("Epoch")
plt.ylabel(metric_key)
plt.title(
    f"{metric_key} over epochs for models with final {metric_key} < {only_models_below} "
    f"(final epoch {final_epoch}, n={len(selected_models)} of {n_models})"
)
plt.grid(True)
plt.tight_layout()

plt.savefig(
    out_dir / f"threshold_{metric_key}_epoch{final_epoch}_thr{only_models_below}.pdf",
    bbox_inches="tight"
)
plt.show()

# ----- table of selected models -----
display(pd.DataFrame({
    "short": [short_labels[m] for m in selected_models],
    "model": selected_models,
    f"final_{metric_key}": final_scores.loc[selected_models].to_numpy(),
}).round(4).reset_index(drop=True))

# ----- epoch stats over the selected models -----
epoch_stats = pd.DataFrame({
    "epoch": acc_sel.columns.astype(int),
    "mean": acc_sel.mean(axis=0).to_numpy(),
    "std": acc_sel.std(axis=0, ddof=1).to_numpy(),
    "min": acc_sel.min(axis=0).to_numpy(),
    "max": acc_sel.max(axis=0).to_numpy(),
}).set_index("epoch")

display(epoch_stats.round(4))

In [ ]:
# ----- optional: plot std over epochs for selected models -----
std_per_epoch = acc_sel.std(axis=0, ddof=1).sort_index()

plt.figure(figsize=fig_big, dpi=180)
plt.plot(std_per_epoch.index.to_numpy(), std_per_epoch.to_numpy(), linewidth=1, marker="o")
plt.xlabel("Epoch")
plt.ylabel(f"{metric_key} std")
plt.title(f"Std of {metric_key} over epochs (final {metric_key} < {only_models_below})")
plt.grid(True)
plt.tight_layout()

plt.savefig(out_dir / f"threshold_std_{metric_key}_epoch{final_epoch}_thr{only_models_below}.pdf",
            bbox_inches="tight")
plt.show()

In [ ]:
only_models_above = 0.09
only_models_below = 0.12
n_last = 15

metric_key = "test_acc"  # or "validation_acc", etc.

# --- ensure numeric ---
dfm = df_long.copy()
dfm[epoch_col] = pd.to_numeric(dfm[epoch_col], errors="coerce")
dfm[metric_key] = pd.to_numeric(dfm[metric_key], errors="coerce")
dfm[metric_key] = dfm[metric_key].replace([-999, -999.0], np.nan)  # optional sentinel

n_models = dfm["model_name"].nunique()

# ----- final epoch -----
final_epoch = int(dfm[epoch_col].max())

# ----- final-epoch score per model -----
final_rows = dfm[dfm[epoch_col] == final_epoch].copy()

# if duplicates at final epoch exist, pick max (or mean)
final_scores = (
    final_rows.groupby("model_name")[metric_key]
    .max()
    .dropna()
)

# ----- select models above threshold (sorted desc) -----
selected_models = (
    final_scores[final_scores < only_models_below]
    .sort_values(ascending=True)
    .index
    .tolist()
)

# ----- build models x epochs matrix for selected models -----
acc_sel = (
    dfm[dfm["model_name"].isin(selected_models)]
    .pivot_table(index="model_name", columns=epoch_col, values=metric_key, aggfunc="max")
    .sort_index(axis=1)
)

# ----- plot curves -----
short_labels = {m: f"m{i + 1:03d}" for i, m in enumerate(selected_models)}

plt.figure(figsize=fig_big, dpi=240)
for m in selected_models:
    plt.plot(acc_sel.columns.to_numpy(), acc_sel.loc[m].to_numpy(), label=short_labels[m])

plt.xlabel("Epoch")
plt.ylabel(metric_key)
plt.title(
    f"{metric_key} over epochs for models with final {metric_key} < {only_models_below} "
    f"(final epoch {final_epoch}, n={len(selected_models)} of {n_models})"
)
plt.grid(True)
plt.tight_layout()

plt.savefig(
    out_dir / f"threshold_{metric_key}_epoch{final_epoch}_thr{only_models_below}.pdf",
    bbox_inches="tight"
)
plt.show()

# ----- table of selected models -----
display(pd.DataFrame({
    "short": [short_labels[m] for m in selected_models],
    "model": selected_models,
    f"final_{metric_key}": final_scores.loc[selected_models].to_numpy(),
}).round(4).reset_index(drop=True))

# ----- epoch stats over the selected models -----
epoch_stats = pd.DataFrame({
    "epoch": acc_sel.columns.astype(int),
    "mean": acc_sel.mean(axis=0).to_numpy(),
    "std": acc_sel.std(axis=0, ddof=1).to_numpy(),
    "min": acc_sel.min(axis=0).to_numpy(),
    "max": acc_sel.max(axis=0).to_numpy(),
}).set_index("epoch")

display(epoch_stats.round(4))

In [ ]:
only_models_above = 0.09
only_models_below = 0.12
n_last = 15

metric_key = "test_acc"  # change if needed

# --- ensure numeric ---
dfm = df_long.copy()
dfm[epoch_col] = pd.to_numeric(dfm[epoch_col], errors="coerce")
dfm[metric_key] = pd.to_numeric(dfm[metric_key], errors="coerce")
dfm[metric_key] = dfm[metric_key].replace([-999, -999.0], np.nan)  # optional sentinel

n_models = dfm["model_name"].nunique()

# --- build full models x epochs matrix from dfm (ALL models) ---
acc_mat = (
    dfm.pivot_table(index="model_name", columns=epoch_col, values=metric_key, aggfunc="max")
    .sort_index(axis=1)
)

# --- last n epochs (global) ---
last_epochs = acc_mat.columns.sort_values()[-n_last:]
final_epoch = int(last_epochs[-1])

# elementwise in-range mask (DataFrame of booleans)
in_range = (acc_mat[last_epochs] >= only_models_above) & (acc_mat[last_epochs] <= only_models_below)

# keep models where ALL last n epochs are in range
mask = in_range.all(axis=1)
selected_models = acc_mat.index[mask]

# optional: sort by last epoch value (desc)
selected_models = (
    acc_mat.loc[selected_models, final_epoch]
    .sort_values(ascending=False)
    .index
    .tolist()
)

acc_sel = acc_mat.loc[selected_models].sort_index(axis=1)
print(f"Selected models: {len(selected_models)}")

# --- plot curves ---
short_labels = {m: f"m{i + 1:03d}" for i, m in enumerate(selected_models)}

plt.figure(figsize=fig_big, dpi=180)
for m in selected_models:
    plt.plot(acc_sel.columns.to_numpy(), acc_sel.loc[m].to_numpy(), label=short_labels[m])

plt.xlabel("Epoch")
plt.ylabel(metric_key)
plt.title(
    f"{metric_key} over epochs for models with last {n_last} epochs in "
    f"[{only_models_above}, {only_models_below}] (n={len(selected_models)} of {n_models})"
)
plt.grid(True)
plt.tight_layout()

plt.savefig(
    out_dir / f"inrange_{metric_key}_last{n_last}_epoch{final_epoch}_"
              f"lo{only_models_above}_hi{only_models_below}.pdf",
    bbox_inches="tight"
)
plt.show()

# --- table ---
display(pd.DataFrame({
    "short": [short_labels[m] for m in selected_models],
    "model": selected_models,
    f"final_{metric_key}": acc_mat.loc[selected_models, final_epoch].to_numpy(),
}).round(4).reset_index(drop=True))

# --- epoch stats over selected models ---
epoch_stats = pd.DataFrame({
    "epoch": acc_sel.columns.astype(int),
    "mean": acc_sel.mean(axis=0).to_numpy(),
    "std": acc_sel.std(axis=0, ddof=1).to_numpy(),
    "min": acc_sel.min(axis=0).to_numpy(),
    "max": acc_sel.max(axis=0).to_numpy(),
}).set_index("epoch")

display(epoch_stats.round(4))

# --- optional: std plot ---
std_per_epoch = acc_sel.std(axis=0, ddof=1).sort_index()

plt.figure(figsize=(10, 4), dpi=180)
plt.plot(std_per_epoch.index.to_numpy(), std_per_epoch.to_numpy(), marker="o")
plt.xlabel("Epoch")
plt.ylabel(f"Std ({metric_key})")
plt.title(
    f"Std over epochs (last {n_last} epochs in [{only_models_above}, {only_models_below}], "
    f"n={len(selected_models)} of {n_models})"
)
plt.grid(True)
plt.tight_layout()

plt.savefig(
    out_dir / f"inrange_std_{metric_key}_last{n_last}_epoch{final_epoch}_"
              f"lo{only_models_above}_hi{only_models_below}.pdf",
    bbox_inches="tight"
)
plt.show()

### Calculate rank switches

In [ ]:
# ---------- helpers (adapted to dfm) ----------
def build_metric_matrix_from_dfm(
        dfm: pd.DataFrame,
        metric_key: str,
        epoch_col: str = "training_iteration",
        model_col: str = "model_name",
        epoch_range: tuple[int, int] | None = None,
        sentinels: tuple[float, ...] = (-999.0, -999),
) -> pd.DataFrame:
    """
    Build model x epoch matrix from dfm (long format).

    Args:
      dfm: long dataframe with columns [model_col, epoch_col, metric_key]
      require_full_epochs: if the complete epochs should be used
      epoch_col: name of the epoch column in the dfm
      epoch_range: range of epochs to be considered
      metric_key: key of the metric
      model_col: name of the model column in the dfm
      sentinels: what should be excluded

    Returns:
      mat: DataFrame index=model, columns=epochs, values=metric
    """
    df = dfm[[model_col, epoch_col, metric_key]].copy()

    df[epoch_col] = pd.to_numeric(df[epoch_col], errors="coerce")
    df[metric_key] = pd.to_numeric(df[metric_key], errors="coerce")
    df[metric_key] = df[metric_key].replace(list(sentinels), np.nan)

    df = df.dropna(subset=[model_col, epoch_col, metric_key])

    # Optionally restrict epochs
    if epoch_range is not None:
        e_min, e_max = epoch_range
        df = df[(df[epoch_col] >= e_min) & (df[epoch_col] <= e_max)]

    # Resolve duplicates per (model, epoch)
    mat = df.pivot_table(
        index=model_col,
        columns=epoch_col,
        values=metric_key,
        aggfunc="max",
    )

    mat = mat.sort_index(axis=1)

    return mat


def rank_per_epoch(mat: pd.DataFrame, higher_is_better: bool = True, tie_method: str = "average"):
    """rank 1 = best. For loss metrics, set higher_is_better=False."""
    return mat.rank(axis=0, ascending=not higher_is_better, method=tie_method)


def rank_change_table(
    mat: pd.DataFrame,
    rank_mat: pd.DataFrame | None = None,
    smoothening_threshold: float | None = None,
):
    if rank_mat is None:
        rank_mat = mat.rank(axis=0, ascending=False, method="average")

    epochs = list(mat.columns)
    if len(epochs) < 2:
        return pd.DataFrame()

    rows = []
    for e0, e1 in zip(epochs[:-1], epochs[1:]):
        r0 = rank_mat[e0]
        r1 = rank_mat[e1]
        diff = r1 - r0  # + => moved down, - => moved up

        row = {
            "from_epoch": e0,
            "to_epoch": e1,
            "models_changed_rank": int((r0 != r1).sum()),
            "models_moved_up": int((diff < 0).sum()),
            "models_moved_down": int((diff > 0).sum()),
            "top1_changed": bool(mat[e0].idxmax() != mat[e1].idxmax()),
        }

        if smoothening_threshold is not None:
            a0 = mat[e0]
            a1 = mat[e1]
            big_move = (a1 - a0).abs() >= smoothening_threshold

            row.update({
                "models_changed_rank_with_smoothening": int(((r0 != r1) & big_move).sum()),
                "models_moved_up_with_smoothening": int(((diff < 0) & big_move).sum()),
                "models_moved_down_with_smoothening": int(((diff > 0) & big_move).sum()),
                "smoothening_threshold_rank_change": smoothening_threshold,
            })

        rows.append(row)

    return pd.DataFrame(rows)


def count_pairwise_flips(acc_t, acc_t1, smoothening_threshold=None):
    a0 = np.asarray(acc_t, dtype=float)
    a1 = np.asarray(acc_t1, dtype=float)
    N = a0.shape[0]

    d0 = a0[:, None] - a0[None, :]
    d1 = a1[:, None] - a1[None, :]

    sign0 = np.sign(d0)
    sign1 = np.sign(d1)

    flipped = (sign0 * sign1) < 0
    mask = np.triu(np.ones((N, N), dtype=bool), k=1)

    if smoothening_threshold is None:
        return int(np.sum(flipped & mask))

    separated = (np.abs(d0) >= smoothening_threshold) & (np.abs(d1) >= smoothening_threshold)
    return int(np.sum(flipped & separated & mask))


def pairwise_flip_table(mat: pd.DataFrame, smoothening_threshold: float = 0.03):
    epochs = list(mat.columns)
    if len(epochs) < 2:
        return pd.DataFrame()

    rows = []
    for e0, e1 in zip(epochs[:-1], epochs[1:]):
        a0 = mat[e0].to_numpy()
        a1 = mat[e1].to_numpy()

        flips_without_smoothening = count_pairwise_flips(a0, a1, smoothening_threshold=None)
        flips_with_smoothening = count_pairwise_flips(a0, a1, smoothening_threshold=smoothening_threshold)

        rows.append({
            "from_epoch": e0,
            "to_epoch": e1,
            "pairwise_flips_without_smoothening": flips_without_smoothening,
            "pairwise_flips_with_smoothening": flips_with_smoothening,
            "smoothening_threshold_pairwise_flip": smoothening_threshold,
        })

    return pd.DataFrame(rows)

In [ ]:
higher_is_better = {
    "train_loss": False,
    "validation_loss": False,
    "test_loss": False,
    "train_acc": True,
    "validation_acc": True,
    "test_acc": True,
}

SMOOTHENING_THRESHOLD = 0.02  # threshold for filtering tiny changes

results = {}

for metric_key in metrics:
    mat = build_metric_matrix_from_dfm(
        dfm=dfm,
        metric_key=metric_key,
        epoch_col=epoch_col,
        model_col=model_col,
        epoch_range=EPOCH_RANGE,
    )

    n_models_metric = mat.shape[0]
    print(f"{metric_key}: models with all epochs {EPOCH_RANGE}: {n_models_metric}")

    if mat.shape[1] < 2 or n_models_metric == 0:
        results[metric_key] = pd.DataFrame()
        continue

    avg_metric = mat.mean(axis=0)

    rmat = rank_per_epoch(mat, higher_is_better=higher_is_better[metric_key], tie_method="average")

    switch_df = rank_change_table(mat, rank_mat=rmat, smoothening_threshold=SMOOTHENING_THRESHOLD)

    flip_df = pairwise_flip_table(mat, smoothening_threshold=SMOOTHENING_THRESHOLD)
    if not flip_df.empty:
        switch_df = switch_df.merge(flip_df, on=["from_epoch", "to_epoch"], how="left")

    switch_df["avg_from"] = switch_df["from_epoch"].map(avg_metric)
    switch_df["avg_to"] = switch_df["to_epoch"].map(avg_metric)
    switch_df["avg_delta"] = switch_df["avg_to"] - switch_df["avg_from"]

    # add metric + n_models as actual columns
    switch_df["metric"] = metric_key
    switch_df["n_models"] = n_models_metric

    # (optional) also keep in attrs if you like
    switch_df.attrs["n_models"] = n_models_metric

    results[metric_key] = switch_df

# one big table across metrics
all_switches = pd.concat([df for df in results.values() if not df.empty], ignore_index=True)
display(all_switches)

Define a method to extract count of models out of switch_df

In [ ]:
def n_models_from_switch_df(switch_df):
    if switch_df is None or switch_df.empty or "n_models" not in switch_df.columns:
        return "?"
    # n_models is constant within the metric's switch_df
    return int(switch_df["n_models"].iloc[0])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=fig_medium, dpi=180, sharex=True)
axes = axes.ravel()

for ax, metric_key in zip(axes, metrics):
    switch_df = results.get(metric_key, None)
    N = n_models_from_switch_df(switch_df)

    if switch_df is None or switch_df.empty:
        ax.set_title(f"{metric_key} (no data)")
        ax.grid(True)
        continue

    x = switch_df["to_epoch"]

    # read smoothening threshold from df if present
    threshold_label = None
    if "smoothening_threshold_rank_change" in switch_df.columns and not switch_df["smoothening_threshold_rank_change"].isna().all():
        threshold_label = switch_df["smoothening_threshold_rank_change"].iloc[0]

    ax.plot(x, switch_df["models_changed_rank"], label="without smoothening")

    if "models_changed_rank_with_smoothening" in switch_df.columns:
        if threshold_label is not None:
            ax.plot(x, switch_df["models_changed_rank_with_smoothening"], label=f"with smoothening (threshold={threshold_label:g})")
        else:
            ax.plot(x, switch_df["models_changed_rank_with_smoothening"], label="with smoothening")


    ax.set_title(f"{metric_key} (N={N})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("# models changed rank")
    ax.grid(True)
    ax.legend(fontsize=9)

plt.suptitle("Rank switches across epochs (with vs without smoothening)", y=1.02)
plt.tight_layout()

plt.savefig(out_dir / "rank_switches_all_metrics.pdf", bbox_inches="tight")
plt.show()

Pairwise flips

In [ ]:
# ---------------------------
# Figure 1: pairwise flips (without smoothening)
# ---------------------------
fig, axes = plt.subplots(2, 3, figsize=fig_medium, dpi=180, sharex=True)
axes = axes.ravel()

for ax, metric_key in zip(axes, metrics):
    switch_df = results.get(metric_key, None)
    N = n_models_from_switch_df(switch_df)

    if switch_df is None or switch_df.empty or "pairwise_flips_without_smoothening" not in switch_df.columns:
        ax.set_title(f"{metric_key} (N={N}) - no data")
        ax.grid(True)
        continue

    x = switch_df["to_epoch"]
    ax.plot(x, switch_df["pairwise_flips_without_smoothening"])

    ax.set_title(f"{metric_key} (N={N})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("# pairwise flips")
    ax.grid(True)

plt.suptitle("Pairwise order flips across epochs (without smoothening)", y=1.02)
plt.tight_layout()
plt.savefig(out_dir / "pairwise_flips_all_metrics_without_smoothening.pdf", bbox_inches="tight", facecolor="white")
plt.show()

# ---------------------------
# Figure 2: pairwise flips (with smoothening)
# ---------------------------
fig, axes = plt.subplots(2, 3, figsize=(14, 7), dpi=180, sharex=True)
axes = axes.ravel()

threshold_label = None  # show first threshold we find (assuming a global value)
for ax, metric_key in zip(axes, metrics):
    switch_df = results.get(metric_key, None)
    N = n_models_from_switch_df(switch_df)

    if switch_df is None or switch_df.empty or "pairwise_flips_with_smoothening" not in switch_df.columns:
        ax.set_title(f"{metric_key} (N={N}) - no data")
        ax.grid(True)
        continue

    x = switch_df["to_epoch"]

    if threshold_label is None and "smoothening_threshold_pairwise_flip" in switch_df.columns and not switch_df[
        "smoothening_threshold_pairwise_flip"].isna().all():
        threshold_label = switch_df["smoothening_threshold_pairwise_flip"].iloc[0]

    ax.plot(x, switch_df["pairwise_flips_with_smoothening"])

    ax.set_title(f"{metric_key} (N={N})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("# pairwise flips")
    ax.grid(True)

if threshold_label is not None:
    plt.suptitle(f"Pairwise order flips across epochs (with smoothening, threshold={threshold_label:g})", y=1.02)
else:
    plt.suptitle("Pairwise order flips across epochs (with smoothening)", y=1.02)

plt.tight_layout()
plt.savefig(out_dir / "pairwise_flips_all_metrics_with_smoothening.pdf", bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
for metric_key in metrics:
    switch_df = results.get(metric_key, None)
    if switch_df is None or switch_df.empty:
        print(f"{metric_key}: no data")
        continue

    x = switch_df["to_epoch"]
    N = n_models_from_switch_df(switch_df)
    y_delta = switch_df["avg_delta"]

    # constrained_layout avoids overlapping labels/titles
    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=(12, 4.5),
        dpi=180,
        constrained_layout=True
    )

    # Plot 1: avg metric delta over epochs
    ax1.plot(x, y_delta)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel(f"Avg Δ {metric_key} (to - from)")
    ax1.set_title(f"Average {metric_key} delta per epoch (N={N})", pad=12)
    ax1.grid(True)

    # Plot 2: relationship scatter
    ax2.scatter(y_delta, switch_df["models_changed_rank"], s=14)
    ax2.set_xlabel(f"Avg Δ {metric_key}")
    ax2.set_ylabel("Models changed rank")
    ax2.set_title(f"{metric_key} jumps vs rank changes (N={N})", pad=12)
    ax2.grid(True)

    fig.savefig(out_dir / f"avg_delta_vs_rankchange_{metric_key}.pdf",
                bbox_inches="tight", facecolor="white")
    plt.show()

### Find late bloomers

In [ ]:
def rank_per_epoch_metric(mat: pd.DataFrame, metric_key: str) -> pd.DataFrame:
    """Rank 1 = best. For losses, best = lowest => ascending=True."""
    asc = not higher_is_better[metric_key]  # loss => True, acc => False
    return mat.rank(axis=0, ascending=asc, method="average")


def count_late_bloomers_by_rank_percentile(
        mat: pd.DataFrame,
        metric_key: str,
        epoch_mid: int,
        epoch_end: int,
        bottom_frac_mid: float = 0.30,
        top_frac_end: float = 0.20,
        return_models: bool = True,
        return_table: bool = False,
):
    """
    Late bloomer: bottom `bottom_frac_mid` at epoch_mid, top `top_frac_end` at epoch_end.
    Rank convention: 1 = best (for acc: highest; for loss: lowest).
    """
    if epoch_mid not in mat.columns or epoch_end not in mat.columns:
        raise ValueError(f"{metric_key}: mat must contain columns {epoch_mid} and {epoch_end}")

    # ranks (1 = best)
    rank_mat = rank_per_epoch_metric(mat, metric_key)

    n_models = mat.shape[0]
    if n_models == 0:
        out = [0]
        if return_models: out.append([])
        if return_table: out.append(pd.DataFrame())
        return tuple(out)

    # thresholds in rank units
    bottom_start_rank = int(np.floor((1.0 - bottom_frac_mid) * n_models)) + 1  # bottom X%
    top_end_rank = int(np.ceil(top_frac_end * n_models))  # top Y%

    r_mid = rank_mat[epoch_mid]
    r_end = rank_mat[epoch_end]

    mask = (r_mid >= bottom_start_rank) & (r_end <= top_end_rank)

    models = r_mid.index[mask].tolist()
    count = len(models)

    out = [count]
    if return_models:
        out.append(models)

    if return_table:
        # use neutral names: "value" rather than "acc"
        tbl = pd.DataFrame({
            "rank_epoch_mid": r_mid,
            "rank_epoch_end": r_end,
            f"{metric_key}_epoch_mid": mat[epoch_mid],
            f"{metric_key}_epoch_end": mat[epoch_end],
        }).loc[models].sort_values(["rank_epoch_end", "rank_epoch_mid"])
        out.append(tbl)

    return tuple(out) if len(out) > 1 else count

In [ ]:
epoch_mid = 15
epoch_end = 50
bottom_frac_mid = 0.30
top_frac_end = 0.20

late_bloomers = {}  # metric -> dict with count/models/table

for metric_key in metrics:
    mat = build_metric_matrix_from_dfm(
        dfm=dfm,
        metric_key=metric_key,
        epoch_col="training_iteration",
        model_col="model_name",
        epoch_range=(1, epoch_end),  # ensure epoch_end exists
    )

    if mat.empty or epoch_mid not in mat.columns or epoch_end not in mat.columns:
        late_bloomers[metric_key] = {"count": 0, "models": [], "table": pd.DataFrame()}
        print(f"{metric_key}: no data / missing epochs")
        continue

    count, models, tbl = count_late_bloomers_by_rank_percentile(
        mat,
        metric_key=metric_key,
        epoch_mid=epoch_mid,
        epoch_end=epoch_end,
        bottom_frac_mid=bottom_frac_mid,
        top_frac_end=top_frac_end,
        return_models=True,
        return_table=True,
    )

    late_bloomers[metric_key] = {"count": count, "models": models, "table": tbl}

    print(f"{metric_key}: late bloomers = {count} (N={mat.shape[0]})")
    display(tbl.head(20))

# Optional: summary table across metrics
summary_tbl = pd.DataFrame([
    {"metric": m, "late_bloomers": late_bloomers[m]["count"]}
    for m in metrics
]).sort_values("late_bloomers", ascending=False)

display(summary_tbl)

## Cluster Analysis
To better understand if it is right, that model which are bad in the beginning are also bad in the end I will cluster these models and see which belong together.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

#### Per-model learning-curve features (from `acc_mat`)

- `final_acc`: accuracy at the last epoch (e.g., epoch 50). Chosen as the standard endpoint/leaderboard performance.
- `auc_norm`: area under the accuracy–epoch curve (trapezoidal integral), normalized by epoch span. Chosen to capture anytime performance and training efficiency (good early vs only good late).
- `peak_acc`: maximum accuracy reached at any epoch. Chosen to identify models that can briefly reach high performance even if they don’t finish there.
- `peak_epoch`: epoch where `peak_acc` occurs. Chosen to see whether best performance happens early vs late and to inform early-stopping behavior.
- `peak_drop`: `peak_acc - final_acc` (drop from best checkpoint to final). Chosen as a simple proxy for overfitting/late degradation.
- `mean_delta`: mean per-epoch improvement, average of `acc(t+1) - acc(t)`. Chosen to summarize typical learning speed/progress rate.
- `std_delta`: standard deviation of per-epoch improvements `acc(t+1) - acc(t)`. Chosen to quantify volatility/instability in learning.
- `std_acc`: standard deviation of accuracy values across epochs. Chosen to capture overall curve variability (flat vs strongly changing/fluctuating).
- `early_slope`: slope from start to an early cutoff (default ~first 10 epochs), `(acc(epoch_k) - acc(epoch_1)) / (epoch_k - epoch_1)`. Chosen to quantify early learning speed (fast starters vs slow starters).
- `late_slope`: slope over the last part of training (default ~last 10 epochs), `(acc(final) - acc(epoch_{T-k})) / (epoch_final - epoch_{T-k})`. Chosen to quantify continued improvement vs saturation and highlight late bloomers.
- `t95_epoch`: first epoch where `acc >= 0.95 * final_acc`. Chosen as an interpretable time-to-near-final-performance metric (convergence speed).
- `avg_regret`: average gap to final accuracy, mean of `final_acc - acc(t)` across epochs. Chosen as another efficiency metric: low regret means near-final performance most of training; high regret means long ramp-up.

In [ ]:
def extract_curve_features(
        mat: pd.DataFrame,
        smooth_window: int = 1,
        t95: float = 0.95,
        early_epochs: int = 10,
        late_epochs: int = 10,
) -> pd.DataFrame:
    """
    Compute per-model curve features from mat (index=model, columns=epochs).
    Works for both accuracy and loss (it's just numbers).
    """
    mat = mat.sort_index(axis=1)
    epochs = mat.columns.to_numpy()
    E = len(epochs)

    X = mat.to_numpy(dtype=float)

    # optional smoothing (moving average)
    if smooth_window and smooth_window > 1:
        w = smooth_window
        pad = w // 2
        Xpad = np.pad(X, ((0, 0), (pad, pad)), mode="edge")
        Xs = np.empty_like(X)
        kernel = np.ones(w) / w
        for i in range(X.shape[0]):
            Xs[i] = np.convolve(Xpad[i], kernel, mode="valid")
        X_use = Xs
    else:
        X_use = X

    final_val = X_use[:, -1]
    peak_val = np.max(X_use, axis=1)
    peak_epoch_idx = np.argmax(X_use, axis=1)
    peak_epoch = epochs[peak_epoch_idx]

    peak_drop = peak_val - final_val

    auc = np.trapezoid(X_use, x=epochs, axis=1)
    auc_norm = auc / (epochs[-1] - epochs[0] + 1e-12)

    dX = np.diff(X_use, axis=1)
    mean_delta = dX.mean(axis=1)
    std_delta = dX.std(axis=1)
    std_val = X_use.std(axis=1)

    kE = min(early_epochs, E - 1)
    kL = min(late_epochs, E - 1)

    early_slope = (X_use[:, kE] - X_use[:, 0]) / (epochs[kE] - epochs[0] + 1e-12)
    late_slope = (X_use[:, -1] - X_use[:, -(kL + 1)]) / (epochs[-1] - epochs[-(kL + 1)] + 1e-12)

    # time-to-95% of final (first epoch where value >= 0.95*final)
    thresh = t95 * final_val
    t95_epoch = np.full(X_use.shape[0], np.nan, dtype=float)
    for i in range(X_use.shape[0]):
        hit = np.where(X_use[i] >= thresh[i])[0]
        if len(hit) > 0:
            t95_epoch[i] = float(epochs[hit[0]])

    avg_regret = np.mean(final_val[:, None] - X_use, axis=1)

    feat_df = pd.DataFrame({
        "final_val": final_val,
        "auc_norm": auc_norm,
        "peak_val": peak_val,
        "peak_epoch": peak_epoch.astype(float),
        "peak_drop": peak_drop,
        "mean_delta": mean_delta,
        "std_delta": std_delta,
        "std_val": std_val,
        "early_slope": early_slope,
        "late_slope": late_slope,
        "t95_epoch": t95_epoch,
        "avg_regret": avg_regret,
    }, index=mat.index)

    return feat_df


def cluster_models_by_features(
        feat_df: pd.DataFrame,
        k: int = 5,
        use_cols=None,
        random_state: int = 0,
):
    if use_cols is None:
        use_cols = [
            "final_val", "auc_norm", "peak_drop",
            "std_delta", "early_slope", "late_slope",
            "t95_epoch", "avg_regret",
        ]

    feat_used = feat_df[use_cols].copy()
    feat_used = feat_used.apply(lambda c: c.fillna(c.median()), axis=0)

    scaler = StandardScaler()
    Z = scaler.fit_transform(feat_used.to_numpy())

    kmeans = KMeans(n_clusters=k, n_init=20, random_state=random_state)
    y = kmeans.fit_predict(Z)

    sil = np.nan
    if k >= 2 and k < len(feat_used):
        try:
            sil = silhouette_score(Z, y)
        except Exception:
            sil = np.nan

    labels = pd.Series(y, index=feat_df.index, name="cluster")
    return labels, feat_used, scaler, kmeans, sil


def choose_k_by_silhouette(feat_df, k_min=2, k_max=10, use_cols=None, random_state=0):
    rows = []
    for k in range(k_min, k_max + 1):
        labels, feat_used, scaler, kmeans, sil = cluster_models_by_features(
            feat_df, k=k, use_cols=use_cols, random_state=random_state
        )
        rows.append({"k": k, "silhouette": sil})
    return pd.DataFrame(rows).sort_values("k")


def plot_cluster_median_curves(
        mat: pd.DataFrame,
        labels: pd.Series,
        ylabel: str,
        title: str,
        figsize=(10, 6),
        dpi=180,
        show_median=True,
        show_iqr=True,
        show_mean=False,
        show_std=False,
        min_models_per_cluster=1,
        save_path=None,
):
    mat = mat.sort_index(axis=1)
    epochs = mat.columns.to_numpy()

    cluster_sizes = labels.value_counts().sort_values(ascending=False)
    cluster_order = cluster_sizes.index.tolist()

    plt.figure(figsize=figsize, dpi=dpi)
    for c in cluster_order:
        members = labels.index[labels == c]
        if len(members) < min_models_per_cluster:
            continue

        M = mat.loc[members].to_numpy(dtype=float)
        med = np.median(M, axis=0)
        mean = np.mean(M, axis=0)
        std = np.std(M, axis=0)

        if show_median:
            plt.plot(epochs, med, linewidth=2, label=f"cluster {c} median (n={len(members)})")

        if show_mean:
            plt.plot(epochs, mean, linestyle="--", linewidth=1.8, label=f"cluster {c} mean (n={len(members)})")

        if show_iqr:
            q25 = np.quantile(M, 0.25, axis=0)
            q75 = np.quantile(M, 0.75, axis=0)
            plt.fill_between(epochs, q25, q75, alpha=0.15)

        if show_std:
            plt.fill_between(epochs, mean - std, mean + std, alpha=0.10)

    plt.xlabel("epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
LOSS_MAX = 10.0
loss_metrics = {"train_loss", "validation_loss", "test_loss"}

def build_metric_matrix_with_metric_specific_loss_filter(
        dfm: pd.DataFrame,
        metric_key: str,
        epoch_col: str = "training_iteration",
        model_col: str = "model_name",
        epoch_range=(1, 50),
        loss_max=10.0,
):
    """
    Build model×epoch matrix for metric_key.
    If metric_key is a loss metric, drop models ONLY for this metric where max(loss)>loss_max.
    Accuracy metrics are never filtered.
    """
    d = dfm.copy()

    # numeric conversion + sentinel handling
    d[epoch_col] = pd.to_numeric(d[epoch_col], errors="coerce")
    d[metric_key] = pd.to_numeric(d[metric_key], errors="coerce").replace([-999, -999.0], np.nan)

    # metric-specific filtering (loss only)
    if metric_key in loss_metrics:
        max_per_model = d.groupby(model_col)[metric_key].max()
        bad_models = max_per_model.index[max_per_model > loss_max]
        d = d[~d[model_col].isin(bad_models)].copy()
        print(f"{metric_key}: dropped {len(bad_models)} models with {metric_key} > {loss_max}")
    else:
        print(f"{metric_key}: no loss filtering (accuracy metric)")

    # build matrix (your existing function)
    mat = build_metric_matrix_from_dfm(
        dfm=d,
        metric_key=metric_key,
        epoch_col=epoch_col,
        model_col=model_col,
        epoch_range=epoch_range,
    )
    return mat

In [ ]:
#  Optional: if you want "higher is better" for losses too, flip sign:
flip_losses = False  # set True to cluster on (-loss) so "bigger is better"

loss_metrics = {"train_loss", "validation_loss", "test_loss"}

cluster_outputs = {}  # metric -> dict with feat_df, labels, sil_table, chosen_k, mat_used

for metric_key in metrics:
    mat = build_metric_matrix_with_metric_specific_loss_filter(
        dfm=dfm,
        metric_key=metric_key,
        epoch_col=epoch_col,
        model_col=model_col,
        epoch_range=EPOCH_RANGE,
        loss_max=10.0,
    )

    if mat.empty or mat.shape[1] < 2:
        print(f"{metric_key}: no data")
        continue

    mat_use = mat.copy()
    if flip_losses and metric_key in loss_metrics:
        mat_use = -mat_use  # so curves go "up" when loss decreases

    # --- features ---
    feat_df = extract_curve_features(
        mat_use,
        smooth_window=3,
        t95=0.95,
        early_epochs=10,
        late_epochs=10,
    )

    # --- pick k via silhouette (optional) ---
    sil_table = choose_k_by_silhouette(feat_df, k_min=2, k_max=10)
    display(sil_table)

    chosen_k = int(sil_table.loc[sil_table["silhouette"].idxmax(), "k"])
    # or override:
    # chosen_k = 5

    labels, feat_used, scaler, kmeans, sil = cluster_models_by_features(feat_df, k=chosen_k)

    print(f"{metric_key}: chosen k={chosen_k}, silhouette={sil:.3f}, n_models={len(feat_df)}")

    # show small table
    final_feature_table = feat_df.join(labels).sort_values(["cluster", "final_val"], ascending=[True, False])
    to_show = final_feature_table.head(20).rename_axis("model").reset_index()
    to_show["model"] = to_show["model"].astype(str).str[:40]
    display(to_show)

    # --- plots ---
    ylabel = metric_key if not (flip_losses and metric_key in loss_metrics) else f"-{metric_key}"
    plot_cluster_median_curves(
        mat_use,
        labels,
        ylabel=ylabel,
        title=f"{metric_key}: median curves per cluster with IQR (k={chosen_k}, n={len(feat_df)})",
        figsize=fig_medium,
        save_path=out_dir / f"cluster_{metric_key}_median_IQR_k{chosen_k}.pdf",
        show_median=True,
        show_iqr=True,
        show_mean=False,
        show_std=False,
    )

    plot_cluster_median_curves(
        mat_use,
        labels,
        ylabel=ylabel,
        title=f"{metric_key}: mean curves per cluster with std (k={chosen_k}, n={len(feat_df)})",
        figsize=fig_medium,
        save_path=out_dir / f"cluster_{metric_key}_mean_std_k{chosen_k}.pdf",
        show_median=False,
        show_iqr=False,
        show_mean=True,
        show_std=True,
    )

    cluster_outputs[metric_key] = {
        "mat": mat_use,
        "feat_df": feat_df,
        "labels": labels,
        "sil_table": sil_table,
        "k": chosen_k,
        "silhouette": sil,
    }